In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
ANALYSIS_DIR = Path('data/processed/analysis')
OUTPUT_TABLES = Path('reports/descriptives')
OUTPUT_FIGURES = Path('reports/figures')

OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURES.mkdir(parents=True, exist_ok=True)

WIDE_FILE = ANALYSIS_DIR / 'analysis_dataset_wide.csv'
LONG_FILE = ANALYSIS_DIR / 'all_signals_long.csv'

PRIMARY_SIGNALS = [
    't_neck',
    't_chest',
    't_abdomen',
    't_dorsal',
    't_hand',
    't_shin',
    't_microclimatefront',
    't_microclimateback',
    't_environment',
    't_core',
    'rh_neck',
    'rh_chest',
    'rh_abdomen',
    'rh_dorsal',
    'rh_hand',
    'rh_shin',
    'rh_microclimatefront',
    'rh_microclimateback',
    'rh_environment',
    'hr',
    't_microclimate_gradient',
    'rh_microclimate_gradient',
]

PHASE_ORDER = [
    'adaptation_seated',
    'standing_pre',
    'low_1',
    'moderate_1',
    'low_2',
    'moderate_2',
    'standing_post',
]

CONDITION_ORDER = ['no_wind', 'wind_low', 'wind_high']

In [ ]:
def load_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    if not WIDE_FILE.exists():
        raise FileNotFoundError(f'Missing file: {WIDE_FILE}')
    if not LONG_FILE.exists():
        raise FileNotFoundError(f'Missing file: {LONG_FILE}')

    df_wide = pd.read_csv(WIDE_FILE)
    df_long = pd.read_csv(LONG_FILE)

    if 'phase' in df_wide.columns:
        df_wide['phase'] = pd.Categorical(df_wide['phase'], categories=PHASE_ORDER, ordered=True)
    if 'condition' in df_wide.columns:
        df_wide['condition'] = pd.Categorical(df_wide['condition'], categories=CONDITION_ORDER, ordered=True)
    if 'condition' in df_long.columns:
        df_long['condition'] = pd.Categorical(df_long['condition'], categories=CONDITION_ORDER, ordered=True)

    return df_wide, df_long


def get_available_signals(df_wide: pd.DataFrame) -> list[str]:
    return [signal for signal in PRIMARY_SIGNALS if signal in df_wide.columns]


def save_overall_descriptives(df_wide: pd.DataFrame, signals: list[str]) -> None:
    rows = []
    for signal in signals:
        grouped = df_wide.groupby('condition', observed=False)[signal]
        summary = grouped.agg(['count', 'mean', 'std', 'median', 'min', 'max']).reset_index()
        summary.insert(0, 'signal', signal)
        rows.append(summary)

    if rows:
        out = pd.concat(rows, ignore_index=True)
        out.to_csv(OUTPUT_TABLES / 'overall_descriptives_by_condition.csv', index=False)


def save_phase_descriptives(df_wide: pd.DataFrame, signals: list[str]) -> None:
    rows = []
    for signal in signals:
        summary = (
            df_wide.groupby(['condition', 'phase'], observed=False)[signal]
            .agg(['count', 'mean', 'std', 'median', 'min', 'max'])
            .reset_index()
        )
        summary.insert(0, 'signal', signal)
        rows.append(summary)

    if rows:
        out = pd.concat(rows, ignore_index=True)
        out.to_csv(OUTPUT_TABLES / 'phase_descriptives_by_condition.csv', index=False)


def save_subject_phase_means(df_wide: pd.DataFrame, signals: list[str]) -> None:
    group_cols = ['subject_id', 'condition', 'phase']
    cols = [c for c in signals if c in df_wide.columns]
    out = df_wide[group_cols + cols].groupby(group_cols, observed=False).mean(numeric_only=True).reset_index()
    out.to_csv(OUTPUT_TABLES / 'subject_phase_means.csv', index=False)


def save_timebin_means(df_wide: pd.DataFrame, signals: list[str]) -> None:
    if 'time_bin_5min' not in df_wide.columns:
        return
    group_cols = ['condition', 'time_bin_5min']
    cols = [c for c in signals if c in df_wide.columns]
    out = df_wide[group_cols + cols].groupby(group_cols, observed=False).agg(['mean', 'std']).reset_index()
    out.columns = ['_'.join(col).strip('_') if isinstance(col, tuple) else col for col in out.columns]
    out.to_csv(OUTPUT_TABLES / 'timebin_5min_descriptives.csv', index=False)


def plot_signal_over_time(df_wide: pd.DataFrame, signal: str) -> None:
    if signal not in df_wide.columns:
        return

    grouped = (
        df_wide.groupby(['condition', 'elapsed_minutes'], observed=False)[signal]
        .agg(['mean', 'std', 'count'])
        .reset_index()
    )
    grouped['sem'] = grouped['std'] / grouped['count'].pow(0.5)

    plt.figure(figsize=(8, 5))
    for condition in CONDITION_ORDER:
        sub = grouped[grouped['condition'] == condition].sort_values('elapsed_minutes')
        if sub.empty:
            continue
        plt.plot(sub['elapsed_minutes'], sub['mean'], label=condition)
        lower = sub['mean'] - sub['sem'].fillna(0)
        upper = sub['mean'] + sub['sem'].fillna(0)
        plt.fill_between(sub['elapsed_minutes'], lower, upper, alpha=0.2)

    for x in [10, 20, 30, 40, 50, 60]:
        plt.axvline(x=x, linewidth=0.8, linestyle='--')

    plt.xlabel('Elapsed minutes')
    plt.ylabel(signal)
    plt.title(f'{signal} over time')
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_FIGURES / f'{signal}_over_time.png', dpi=300)
    plt.close()


def plot_phase_boxplot(df_wide: pd.DataFrame, signal: str) -> None:
    if signal not in df_wide.columns or 'phase' not in df_wide.columns:
        return

    phase_means = (
        df_wide.groupby(['subject_id', 'condition', 'phase'], observed=False)[signal]
        .mean()
        .reset_index()
    )

    positions = []
    data = []
    labels = []
    pos = 1
    gap = 1

    for phase in PHASE_ORDER:
        for condition in CONDITION_ORDER:
            vals = phase_means.loc[
                (phase_means['phase'] == phase) & (phase_means['condition'] == condition), signal
            ].dropna()
            if vals.empty:
                pos += 1
                continue
            positions.append(pos)
            data.append(vals.values)
            labels.append(f'{phase}\n{condition}')
            pos += 1
        pos += gap

    if not data:
        return

    plt.figure(figsize=(14, 6))
    plt.boxplot(data, positions=positions)
    plt.xticks(positions, labels, rotation=45, ha='right')
    plt.ylabel(signal)
    plt.title(f'{signal} by phase and condition')
    plt.tight_layout()
    plt.savefig(OUTPUT_FIGURES / f'{signal}_by_phase_boxplot.png', dpi=300)
    plt.close()


def export_primary_signal_list(signals: list[str]) -> None:
    pd.DataFrame({'signal': signals}).to_csv(OUTPUT_TABLES / 'available_signals.csv', index=False)


def main() -> None:
    df_wide, df_long = load_data()
    signals = get_available_signals(df_wide)

    if not signals:
        raise ValueError('No expected signals found in analysis_dataset_wide.csv')

    export_primary_signal_list(signals)
    save_overall_descriptives(df_wide, signals)
    save_phase_descriptives(df_wide, signals)
    save_subject_phase_means(df_wide, signals)
    save_timebin_means(df_wide, signals)

    key_plot_signals = [
        't_dorsal',
        't_microclimateback',
        't_microclimatefront',
        't_microclimate_gradient',
        'rh_dorsal',
        'rh_microclimateback',
        'rh_microclimatefront',
        'rh_microclimate_gradient',
        't_core',
        'hr',
    ]

    for signal in key_plot_signals:
        if signal in signals:
            plot_signal_over_time(df_wide, signal)
            plot_phase_boxplot(df_wide, signal)

    print('Descriptive analysis completed successfully.')
    print(f'Tables saved in: {OUTPUT_TABLES}')
    print(f'Figures saved in: {OUTPUT_FIGURES}')


if __name__ == '__main__':
    main()
